# PartaGPU — Distributed multi-GPU training

This notebook shows how to use PartaGPU to leverage the GPUs of several machines in a classroom.

## Prerequisites

1. **The PartaGPU application** must be running on this machine (and on the machines that share their resources).
2. **A room** must be configured (the same access code must be entered on every machine).
3. **Sharing** must be enabled on the remote machines.
4. The Python package is installed via `pip install partagpu`.

## Setting up an isolated Python environment (venv)

To avoid polluting the system Python, this `examples/` folder uses a local **venv** (not tracked by Git).

### First-time setup — create the venv

From a terminal, at the project root:

```bash
cd examples
python3 -m venv venv
source venv/bin/activate          # Linux or macOS
# .\venv\Scripts\activate         # Windows PowerShell
pip install -r requirements.txt
pip install -e ../python          # the partagpu package, in editable mode
python -m ipykernel install --user --name=partagpu-examples --display-name="Python (PartaGPU examples)"
```

The `partagpu` package is installed **in editable mode from `../python`**: while the API still evolves, you automatically pick up the latest version on every `git pull`. Once the package stabilises it will be published on PyPI and a plain `pip install partagpu` will be enough.

### Every time you open the notebook

1. Open this file in Jupyter or VS Code.
2. In the top-right corner, **select the kernel** `Python (PartaGPU examples)`.
3. That's it — every cell will run inside the venv.

### Check that the venv is the active one

Run the cell below: `sys.executable` must point to `examples/venv/bin/python`.

### The venv is not tracked by Git

The `examples/venv/` folder is listed in `.gitignore`. Each user recreates their own with the commands above.

In [ ]:
import sys
print("Python in use :", sys.executable)
print("Version       :", sys.version.split()[0])

if "examples/venv" in sys.executable.replace("\\", "/"):
    print("OK — you are running inside the local venv.")
else:
    print("Warning: the active kernel is NOT the local venv.")
    print("Select the 'Python (PartaGPU examples)' kernel in the top-right corner.")

## 1. Installing the package

In [ ]:
# Editable install from the cloned repo: the package always reflects the
# current state of your git checkout (handy while the API evolves).
# The path '../python' is relative to the 'examples/' folder.
%pip install -e ../python numpy torch
%load_ext autoreload
%autoreload 2

## 2. Checking the connection to PartaGPU

The package talks to the PartaGPU application through a local HTTP API (`localhost:7654`).

In [ ]:
import requests

try:
    r = requests.get("http://127.0.0.1:7654/api/status", timeout=2)
    print("PartaGPU is running.")
    print(f"Status: {r.json()}")
except requests.ConnectionError:
    print("PartaGPU is not running. Please open the application first.")

## 3. Discovering the machines on the network

This section lists every machine running PartaGPU on the local network.

In [ ]:
from partagpu.discover import get_peers

peers = get_peers()
print(f"{len(peers)} machine(s) detected on the network:\n")

for p in peers:
    status = "Verified" if p.verified else "Not verified"
    sharing = "Active" if p.sharing_enabled else "Inactive"
    print(f"  {p.display_name} ({p.hostname})")
    print(f"    IP: {p.ip} | Auth: {status} | Sharing: {sharing}")
    print(f"    CPU: {p.cpu_limit}% | RAM: {p.ram_limit} MB | GPU: {p.gpu_limit}%")
    print()

## 4. Discovering available GPUs

This is the main entry point: it returns the list of usable GPUs (yours plus those of verified peers that are sharing).

In [ ]:
import partagpu

gpus = partagpu.discover()

print(f"{len(gpus)} GPU(s) available for training:\n")
for g in gpus:
    print(f"  {g}")

if len(gpus) == 0:
    print("No GPU detected.")
    print("Check that:")
    print("  - you have an NVIDIA GPU with the drivers installed;")
    print("  - the remote machines have enabled sharing;")
    print("  - you are in the same room.")

## 5. Local training (on your GPU)

This section trains a **CNN** (about 5 million parameters) on synthetic 3×128×128 images, **using only this machine's GPU**. It is the baseline: the same example, but purely local. To distribute the computation to the rest of the room's GPUs, see the next section.

On an RTX 3060, expect about 12 seconds per epoch, i.e. roughly **2 minutes** in total. Use `Ctrl+C` or the stop button to interrupt the run.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Check that CUDA is available
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Local GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# === Synthetic data: 3x128x128 images, 10 classes ===
N = 4096
X = torch.randn(N, 3, 128, 128)
y = torch.randint(0, 10, (N,))
loader = DataLoader(
    TensorDataset(X, y),
    batch_size=64,
    shuffle=True,
    pin_memory=(device.type == "cuda"),
)

# === ~10M-parameter CNN: 4 conv-BN-ReLU blocks + classifier ===
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        def block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )
        self.features = nn.Sequential(
            block(3, 64),     # 128 -> 64
            block(64, 128),   # 64  -> 32
            block(128, 256),  # 32  -> 16
            block(256, 512),  # 16  -> 8
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = CNN().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params/1e6:.1f}M\n")

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

EPOCHS = 10
print(f"Training for {EPOCHS} epochs (Ctrl+C to interrupt)\n")

start = time.time()
try:
    for epoch in range(EPOCHS):
        epoch_start = time.time()
        total_loss, n_batches = 0.0, 0
        for batch_X, batch_y in loader:
            batch_X = batch_X.to(device, non_blocking=True)
            batch_y = batch_y.to(device, non_blocking=True)

            optimizer.zero_grad()
            output = model(batch_X)
            loss = criterion(output, batch_y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            n_batches += 1

        if device.type == "cuda":
            torch.cuda.synchronize()
        dt = time.time() - epoch_start
        print(f"Epoch {epoch+1:2d}/{EPOCHS} | loss {total_loss/n_batches:.4f} | {dt:5.1f}s")
except KeyboardInterrupt:
    print("\nInterrupted by user.")

print(f"\nTotal: {time.time()-start:.1f}s")


## 6. Remote execution

There are two levels of usage, from the simplest to the most powerful:

- **`partagpu.run_remote(peer, args)`** runs ONE command on ONE peer. Useful to test a peer, fetch a one-off computation, or launch a batch job.
- **`partagpu.distribute(script, args=)`** runs the same PyTorch script on **every GPU in the room in parallel**, with automatic DDP rendezvous. This is the heart of PartaGPU: N machines training together.

In both cases, the PartaGPU application takes care of:

- **HMAC authentication** between peers (derived from the shared room code);
- **sandboxed execution** (bubblewrap, `partagpu` user, command allowlist) on the target machine;
- **result reporting** (stdout, stderr, exit code).

**Prerequisites on every target machine**:

1. it must be in the **same PartaGPU room** (same access code);
2. **sharing must be active** ("Active" badge in the application);
3. `bubblewrap` must be installed (`sudo apt install bubblewrap`);
4. for DDP, `torch` must be available on the peer's system Python (`/usr/bin/python3 -c 'import torch'`).

### 6.1 One command on one peer — `run_remote`

In [ ]:
import partagpu

# GPUs exposed by the room (yours plus those of sharing peers).
gpus = partagpu.discover()
remote_peers = [g for g in gpus if g.host != "local"]

if not remote_peers:
    print("No remote peer is available.")
    print("Check that:")
    print("  - the other machine is in the same PartaGPU room;")
    print("  - sharing is active on the other machine ('Active' badge).")
else:
    target = remote_peers[0]
    print(f"Target: {target}\n")

    # Small Python script that runs on the remote peer. It queries its own
    # GPU and reports back. We use a triple-quoted string to get real
    # newlines: python3 -c '...' accepts compound statements (try/if/...)
    # as long as they are on their own line.
    code = """\
import socket, sys
print('hostname:', socket.gethostname())
print('python  :', sys.version.split()[0])
try:
    import torch
    print('torch   :', torch.__version__)
    print('cuda    :', torch.cuda.is_available())
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print('gpu     :', p.name)
        print('vram_gb :', round(p.total_memory / 1e9, 1))
except ImportError:
    print('torch is not installed on the peer')
"""

    result = partagpu.run_remote(
        target,
        ["python3", "-c", code],
        timeout=60,
    )

    print("=== remote output ===")
    print(result.stdout)
    if result.stderr:
        print("=== stderr ===")
        print(result.stderr)
    print(f"\nstatus={result.status}  exit_code={result.exit_code}")

### 6.2 Multi-machine DDP training — `distribute`

We send the **same script** to **every GPU** in the room (including several GPUs on the same machine, if any). Each rank receives the standard DDP environment variables:

- `MASTER_ADDR` and `MASTER_PORT`: rendezvous endpoint (rank 0);
- `RANK` and `WORLD_SIZE`: position in the global group;
- `LOCAL_RANK`: always `0` from the script's point of view (each worker sees a single GPU through `CUDA_VISIBLE_DEVICES`);
- `PARTAGPU_LOCAL_RANK`: position relative to other workers on the same host (useful for logs);
- `CUDA_VISIBLE_DEVICES`: the physical GPU assigned to this worker;
- `BACKEND`: `nccl` (default on GPU) or `gloo` (CPU).

On the script side:

```python
import torch.distributed as dist
dist.init_process_group(backend=os.environ['BACKEND'], init_method='env://')
device = torch.device('cuda:0')   # a single GPU is visible per worker
```

**Per-machine multi-GPU**: if a machine in the room has 4 GPUs, `partagpu.discover()` returns 4 entries for it (same `host` and `ip`, but `device_index` from 0 to 3) and `distribute` launches 4 concurrent workers on that machine. The total `world_size` is the sum of GPUs across all machines.

The `ddp_train_demo.py` script (next to this notebook) trains a tiny CNN on synthetic data. PartaGPU **transfers the file** into the sandbox of every peer before executing it (`workspace=` parameter).

In [ ]:
import partagpu
from pathlib import Path

# The script lives next to this notebook.
script = Path("ddp_train_demo.py").resolve()
assert script.is_file(), f"not found: {script}"

gpus = partagpu.discover()
print(f"world_size = {len(gpus)} GPU(s):")
for g in gpus:
    print(f"  - {g}")

# Run the same script on every GPU. Blocks until all of them finish.
results = partagpu.distribute(
    script,
    args=["--epochs", "2", "--samples", "512"],
    timeout=300,
)

for r in results:
    print("\n=== rank on", r.target_machine, f"(exit={r.exit_code}) ===")
    print(r.stdout)
    if r.stderr.strip():
        print("--- stderr ---")
        print(r.stderr)

In [ ]:
# --heavy variant: same script, but with a ~5M-parameter CNN on 128x128
# images. Forward and backward take ~10-20 ms per batch (vs. ~1 ms in the
# light mode), long enough for nvidia-smi pmon to catch the GPU within its
# 1 Hz sampling window so that the GPU column finally rises in the
# PartaGPU UI. Expect 30-60 seconds of execution depending on the machines.

results = partagpu.distribute(
    script,
    args=["--heavy", "--epochs", "3", "--samples", "4096"],
    timeout=600,
)

for r in results:
    print("\n=== rank on", r.target_machine, f"(exit={r.exit_code}) ===")
    print(r.stdout)
    if r.stderr.strip():
        print("--- stderr ---")
        print(r.stderr)

### 6.3 Colorisation UNet — code written from the notebook

`distribute()` ships a **`.py` file** to each peer (current API limitation: in-memory Python code cannot be transmitted). To embed a real training workflow into a notebook (model defined in one cell, dispatch in the next, results in a third), the pattern is:

1. use the Jupyter magic `%%writefile colorization_train.py`, which dumps the cell content into a file next to the notebook;
2. call `partagpu.distribute("colorization_train.py", args=[...])`, which ships the file and runs it on every GPU in the room;
3. read the outputs returned by each rank to plot the loss curves.

If you already have a separate colorisation notebook (for instance [`cnn-colorization`](https://github.com/cesar-lizurey/cnn-colorization)), you can apply the same pattern: extract the training code into a `.py` file via `%%writefile`, then call `distribute()` instead of `model.fit(...)`.

**Example below**: a tiny UNet (~200k parameters) that learns to predict the **AB** channels (LAB color space) from the **L** (luminance) channel, trained on **CIFAR-10** with DDP. It demonstrates:

- loading a dataset **over the network** (CIFAR-10 is downloaded by every peer, which works because `distribute()` automatically sets `network=True`);
- the **skip connections** of a UNet: encoder, bottleneck, decoder, with concatenation of encoder activations;
- `DistributedSampler`, which splits the dataset across ranks (each rank sees `dataset_size / world_size` examples per epoch);
- the automatic gradient AllReduce via NCCL after each `backward()`.

**Prerequisite on every peer**: `torchvision` must be installed in addition to `torch`. Clicking **Install ML toolkit** in *My Sharing* on each peer installs the full stack (`torch`, `torchvision`, `numpy`, `scipy`, `pandas`, `scikit-learn`, `matplotlib`, `pillow`).

**Caveat**: the sandbox destroys `/workspace` at the end of every task, so the **trained model is not repatriated automatically**. To recover the weights, either pickle them as base64 in stdout (impractical past a few MB) or use a shared filesystem (NFS, local MinIO over S3, etc.). For this demo we simply read the loss curves from stdout.

In [ ]:
%%writefile colorization_train.py
"""Colorisation UNet — predicts the AB channels from L on CIFAR-10.

Simple chroma approximation (no scikit-image dependency):
    L  = 0.299 R + 0.587 G + 0.114 B
    A' = R - L
    B' = B - L

The model learns `L -> (A', B')`. To rebuild the image we then apply
`R = L + A'`, `B = L + B'`, `G = (L - 0.299*R - 0.114*B) / 0.587`.

4-level UNet (64-128-256-512), CIFAR-10 upsampled to 128x128, about 8M
parameters and steps of 20-30 ms on an RTX 3060 — long enough to be visible
on nvidia-smi monitors that sample at 1 Hz.

At the end of training, rank 0 saves the weights to /workspace/model.pt so
that the notebook can repatriate them via `distribute(outputs=["model.pt"])`.
"""
import argparse, os, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler
import torchvision
import torchvision.transforms as T


def rgb_to_l_ab(x):
    r, g, b = x[:, 0:1], x[:, 1:2], x[:, 2:3]
    lum = 0.299 * r + 0.587 * g + 0.114 * b
    a = r - lum
    bb = b - lum
    ab = torch.cat([a, bb], dim=1)
    return lum, ab


class DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout),
            nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = DoubleConv(1, 64)
        self.enc2 = DoubleConv(64, 128)
        self.enc3 = DoubleConv(128, 256)
        self.enc4 = DoubleConv(256, 512)
        self.pool = nn.MaxPool2d(2)

        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = DoubleConv(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = DoubleConv(128, 64)
        self.head = nn.Conv2d(64, 2, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        d3 = self.up3(e4)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        return self.head(d1)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--epochs", type=int, default=2)
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--image-size", type=int, default=128)
    parser.add_argument("--data-dir", type=str, default="./data")
    parser.add_argument("--save", type=str, default="model.pt",
                        help="path of the checkpoint saved by rank 0 (relative to /workspace)")
    args = parser.parse_args()

    rank = int(os.environ.get("RANK", "0"))
    world_size = int(os.environ.get("WORLD_SIZE", "1"))
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))

    if torch.cuda.is_available():
        torch.cuda.set_device(local_rank)
        device = torch.device(f"cuda:{local_rank}")
        backend = "nccl"
    else:
        device = torch.device("cpu")
        backend = "gloo"

    if world_size > 1:
        dist.init_process_group(backend=backend, init_method="env://")

    if rank == 0:
        print(f"[rank0] world_size={world_size} device={device} backend={backend} "
              f"image_size={args.image_size} batch={args.batch_size}", flush=True)

    transform = T.Compose([
        T.Resize(args.image_size, antialias=True),
        T.ToTensor(),
    ])
    dataset = torchvision.datasets.CIFAR10(
        root=args.data_dir, train=True, download=True, transform=transform
    )

    sampler = (
        DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True)
        if world_size > 1
        else None
    )
    loader = DataLoader(
        dataset,
        batch_size=args.batch_size,
        sampler=sampler,
        shuffle=(sampler is None),
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
        drop_last=True,
        persistent_workers=True,
    )

    model = UNet().to(device)
    if world_size > 1 and torch.cuda.is_available():
        model = DDP(model, device_ids=[local_rank])
    elif world_size > 1:
        model = DDP(model)

    optim = torch.optim.Adam(model.parameters(), lr=args.lr)
    loss_fn = nn.MSELoss()

    n_params = sum(p.numel() for p in model.parameters())
    if rank == 0:
        print(f"[rank0] model params = {n_params:,}", flush=True)

    for epoch in range(args.epochs):
        if sampler is not None:
            sampler.set_epoch(epoch)
        t0 = time.time()
        running = 0.0
        n_batches = 0
        for images, _ in loader:
            images = images.to(device, non_blocking=True)
            lum, ab = rgb_to_l_ab(images)
            pred = model(lum)
            loss = loss_fn(pred, ab)
            optim.zero_grad(set_to_none=True)
            loss.backward()
            optim.step()
            running += loss.item()
            n_batches += 1
        avg = running / max(1, n_batches)
        dt = time.time() - t0
        tag = f"[rank{rank}]" if world_size > 1 else "[local]"
        print(f"{tag} epoch={epoch+1}/{args.epochs} loss={avg:.5f} "
              f"time={dt:.1f}s steps={n_batches}", flush=True)

    if world_size > 1:
        dist.barrier()

    # Only rank 0 saves the checkpoint, to avoid a race between ranks
    # writing to the same path on the same workspace.
    if rank == 0:
        # Unwrap the DDP wrapper to pickle the raw module weights, which is
        # more portable (no "module." prefix, no implicit world_size).
        underlying = model.module if hasattr(model, "module") else model
        torch.save(
            {
                "state_dict": underlying.state_dict(),
                "image_size": args.image_size,
                "n_params": n_params,
                "epochs_trained": args.epochs,
            },
            args.save,
        )
        print(f"[rank0] checkpoint saved: {args.save} ({n_params:,} params)", flush=True)

    if world_size > 1:
        dist.destroy_process_group()

    if rank == 0:
        print("[rank0] done", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
import partagpu

# distribute(outputs=["model.pt"]) repatriates files produced by each rank.
# By DDP convention, only rank 0 saves a checkpoint, so the .artifacts dict
# is non-empty only on results[0]. Other ranks simply return {} (the file
# does not exist in their workspace).
#
# `local=False` excludes the local machine from auto-discovery. It is handy
# when sharing is not enabled on your own machine and you want to dispatch
# to remote peers only. Without this filter, rank 0 lands locally and gets
# a 403 ("Sharing is not enabled on this machine").
results = partagpu.distribute(
    "colorization_train.py",
    args=["--epochs", "2", "--batch-size", "32"],
    timeout=1800,
    live=True,
    outputs=["model.pt"],
    local=False,
)

print()
print("=== final summary ===")
for rank, r in enumerate(results):
    art = ", ".join(f"{k} ({len(v)//1024} KB)" for k, v in r.artifacts.items())
    print(f"rank {rank} on {r.target_machine}: status={r.status} exit={r.exit_code} artifacts={art or 'none'}")


In [ ]:
# Reload the checkpoint into the local venv so that we can use it from the
# notebook. The `model.pt` file landed in RAM via results[0].artifacts, so
# we deserialise it directly from an io.BytesIO without touching the disk.
#
# We redefine the UNet and DoubleConv classes here rather than importing
# colorization_train.py: that script has a top-level `import torchvision`
# (needed for CIFAR-10 inside the sandbox), but the notebook's local venv
# only ships torch and numpy. torchvision is not required to load weights
# and run inference.
import io
import torch
import torch.nn as nn


class DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout),
            nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = DoubleConv(1, 64)
        self.enc2 = DoubleConv(64, 128)
        self.enc3 = DoubleConv(128, 256)
        self.enc4 = DoubleConv(256, 512)
        self.pool = nn.MaxPool2d(2)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = DoubleConv(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = DoubleConv(128, 64)
        self.head = nn.Conv2d(64, 2, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        d3 = self.up3(e4)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        return self.head(d1)


ckpt_bytes = results[0].artifacts.get("model.pt")
assert ckpt_bytes is not None, "rank 0 did not return model.pt"

ckpt = torch.load(io.BytesIO(ckpt_bytes), map_location="cpu", weights_only=False)
model = UNet()
model.load_state_dict(ckpt["state_dict"])
model.eval()

print(f"Model loaded: {ckpt['n_params']:,} parameters, image_size={ckpt['image_size']}, "
      f"epochs trained={ckpt['epochs_trained']}")

# Quick smoke check: feed a random grayscale image and verify the output shape.
with torch.no_grad():
    L = torch.rand(1, 1, ckpt["image_size"], ckpt["image_size"])
    AB = model(L)
print(f"Inference OK: input L {tuple(L.shape)} -> output AB {tuple(AB.shape)}")


## 7. Using the HTTP API directly

If you integrate PartaGPU into a non-Python tool, you can call the local HTTP API (`localhost:7654`) directly.

The available routes are:

- `GET  /api/peers`: list of machines in the room;
- `GET  /api/gpu`: available GPUs (yours plus those of sharing peers);
- `GET  /api/status`: local sharing status;
- `POST /api/dispatch`: equivalent to `partagpu.run_remote`, in raw HTTP.

In [ ]:
import requests, json

# Peers in the room
peers = requests.get("http://127.0.0.1:7654/api/peers").json()
print("=== Peers ===")
print(json.dumps(peers, indent=2, ensure_ascii=False))

# Available GPUs
gpus = requests.get("http://127.0.0.1:7654/api/gpu").json()
print("\n=== GPUs ===")
print(json.dumps(gpus, indent=2, ensure_ascii=False))

# Dispatch a command on the first remote peer (raw HTTP).
remote = [g for g in gpus if g['host'] != 'local']
if remote:
    r = requests.post(
        "http://127.0.0.1:7654/api/dispatch",
        json={
            "peer_ip": remote[0]['ip'],
            "args": ["python3", "-c", "print('hello from remote')"],
            "timeout_secs": 30,
        },
    )
    print("\n=== /api/dispatch ===")
    print(json.dumps(r.json(), indent=2, ensure_ascii=False))
